# 🃏 Previsão de Preço de Cartas Pokémon TCG
## Parte 3 — Modelagem com CatBoost

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from catboost import CatBoostRegressor
from pathlib import Path

DATA_DIR = Path('data')
df = pd.read_csv(DATA_DIR / 'cards_processed.csv')
print(f'Shape: {df.shape}')
df.head(2)

In [ ]:
# Features
cat_features = ['rarity', 'primary_type', 'set_series', 'price_type', 'supertype']
num_features = ['hp', 'subtypes_count', 'set_printed_total', 'release_year', 'card_age_years', 'pokedex_number']

# Target
target_col = 'log_target_price'

feature_cols = cat_features + [c for c in num_features if c in df.columns]
X = df[feature_cols].copy()
y = df[target_col].copy()

cat_indices = [i for i, c in enumerate(feature_cols) if c in cat_features]
print(f'Features: {feature_cols}')
print(f'Cat features indices: {cat_indices}')

In [ ]:
# Split temporal — cartas mais antigas p/ treino, recentes p/ teste
# (ordem por release_year)
df_sorted = df.sort_values('release_year', na_position='first').reset_index(drop=True)
X_sorted = df_sorted[feature_cols]
y_sorted = df_sorted[target_col]

split = int(len(df_sorted) * 0.8)
X_train, X_test = X_sorted.iloc[:split], X_sorted.iloc[split:]
y_train, y_test = y_sorted.iloc[:split], y_sorted.iloc[split:]
print(f'Treino: {len(X_train)} | Teste: {len(X_test)}')

In [ ]:
model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3,
    loss_function='MAE',
    eval_metric='MAE',
    cat_features=cat_indices,
    verbose=50,
    random_seed=42,
    early_stopping_rounds=30
)

model.fit(X_train, y_train, eval_set=(X_test, y_test))
print(f'\n✅ Melhor iteração: {model.get_best_iteration()}')

In [ ]:
# Predição e métricas
y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_test_orig = df_sorted[target_col].iloc[split:]
y_test_orig = np.expm1(y_test_orig.values)

mae = mean_absolute_error(y_test_orig, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred))
r2 = r2_score(y_test_orig, y_pred)

print(f'\n=== MÉTRICAS (ESCALA ORIGINAL $USD) ===')
print(f'MAE :  ${mae:.2f}')
print(f'RMSE: ${rmse:.2f}')
print(f'R²  :  {r2:.4f}')

In [ ]:
# Comparação visual: Real x Predito
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('CatBoost — Previsão de Preço Pokémon TCG', fontsize=14)

ax = axes[0]
ax.scatter(y_test_orig, y_pred, alpha=0.4, s=10)
ax.plot([0, y_test_orig.max()], [0, y_test_orig.max()], 'r--', lw=1)
ax.set_xlabel('Real ($)')
ax.set_ylabel('Predito ($)')
ax.set_title(f'Real vs Predito (R²={r2:.3f})')

ax = axes[1]
res = y_test_orig - y_pred
ax.hist(res, bins=30, edgecolor='white', color='steelblue')
ax.axvline(0, color='red', ls='--')
ax.set_xlabel('Resíduo ($)')
ax.set_title(f'Resíduos (μ={res.mean():.2f})')

ax = axes[2]
imp = model.get_feature_importance(prettified=True)
imp_sorted = imp.sort_values('Importances', ascending=True)
ax.barh(imp_sorted['Feature Id'], imp_sorted['Importances'], color='steelblue')
ax.set_xlabel('Importância')
ax.set_title('Feature Importance')

plt.tight_layout()
plt.savefig(DATA_DIR / 'catboost_results.png', dpi=120, bbox_inches='tight')
plt.savefig('catboost_results.png', dpi=120, bbox_inches='tight')
plt.close()
print('✅ Gráfico salvo: catboost_results.png')